In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Poland roads × districts → Delta tables + `render_roads_map()`
# MAGIC
# MAGIC **Input:** OSM extract `poland-YYMMDD.osm.pbf` in a Unity Catalog Volume. **Compute:** serverless.
# MAGIC
# MAGIC **What this notebook builds**
# MAGIC
# MAGIC | Layer | Table | Content |
# MAGIC |---|---|---|
# MAGIC | ref | `ref_road_category` | A / S / DK / DW → Polish & English name, line colour, line width, draw order (the "how to render" guide) |
# MAGIC | bronze | `osm_road_way` | every OSM way that carries an A / S / DK / DW ref (full-resolution geometry as WKB) |
# MAGIC | bronze | `osm_admin_area` | assembled boundary polygons for voivodeships (4), powiats (6), gminas (7) |
# MAGIC | silver | `dim_district` | one row per district: TERYT `code`, normalised name key, parent codes, bbox, centre, simplified outline |
# MAGIC | silver | `dim_road` | one row per road (`A4`, `S5`, `DK94`, `DW345`): length, bbox, voivodeships crossed |
# MAGIC | **gold** | **`fact_road_gmina`** | **one row per (road, gmina): the road already clipped to that gmina, merged and simplified** |
# MAGIC
# MAGIC **Why the app side stays fast.** All spatial work (assembling polygons, intersecting ~10⁵ road ways with ~2.5k gminas) happens once, here.
# MAGIC Gminas tile Poland and every gmina code starts with its powiat code (4 digits) and voivodeship code (2 digits), so
# MAGIC "road X inside voivodeship / powiat / gmina Y" is a plain `WHERE road_key = … AND woj_code = …` on a table of a few tens of thousands of rows — no geometry maths at request time.
# MAGIC
# MAGIC **Key design decisions**
# MAGIC * Roads are classified from the OSM `ref` tag, **not** from `highway=*`: Polish expressways are `motorway` (dual carriageway) *or* `trunk` (single), national roads are mostly `primary`, voivodeship roads `primary`/`secondary`/`tertiary`.
# MAGIC   OSM Poland writes DK/DW refs as **bare numbers** (`ref=94`, `ref=345`), motorways/expressways as `A4` / `S5`. Rule: `A#`/`S#` → A/S; bare 1–2 digits → DK; bare 3 digits → DW; explicit `DK 94` / `DW 345` also accepted.
# MAGIC * Districts are keyed by the official **TERYT/TERC** code stored in OSM (`teryt:terc`): `02` voivodeship, `0264` powiat, `0264011` gmina. Names are only used to *look codes up*.
# MAGIC * Geometry is stored as **WKB BINARY** (+ bbox columns) – readable by shapely, GeoPandas, `ST_GeomFromWKB()` in Databricks SQL, and by the Databricks SQL connector from a Databricks App.
# MAGIC
# MAGIC **Run it:** set the widgets (catalog, schema, `pbf_path`), *Run all*. Rough RAM need for the whole of Poland: ~5–8 GB (node-location cache) → fits serverless *Standard* (16 GB); if it ever OOMs switch the notebook to *High memory* in the Environment panel.
# MAGIC
# MAGIC **Licence:** derived from OpenStreetMap → ODbL. Show "© OpenStreetMap contributors" wherever you display the data.

# COMMAND ----------

# MAGIC %pip install -q osmium==4.3.2 shapely==2.2.0 pyproj==3.8.0 folium==0.20.0

# COMMAND ----------

dbutils.library.restartPython()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 0 · Configuration

# COMMAND ----------



In [0]:
%pip install shapely
%pip install pyproj
%pip install osmium
%pip install folium

In [0]:
import os, re, json, shutil, tempfile, time, difflib, unicodedata, warnings
from collections import Counter, defaultdict

import numpy as np
import shapely
import pyproj
import osmium
import folium

In [0]:
def _widget(name, default):
    """Databricks widget when available, otherwise env var (lets the same code run outside a notebook)."""
    try:
        dbutils.widgets.text(name, default)
        return (dbutils.widgets.get(name) or default).strip()
    except NameError:
        return os.environ.get(name.upper(), default)

In [0]:
CATALOG = _widget("catalog", "workspace")
SCHEMA = _widget("schema", "undercover_cars")
PBF_PATH = _widget("pbf_path", "/Volumes/workspace/undercover_cars/undercover_cars/maps/poland-261101.osm.pbf")

# "true": read the PBF and rebuild bronze.  "false": reuse the bronze tables already in Delta (skips the slow PBF pass).
REEXTRACT = _widget("reextract", "true").lower() == "true"

COPY_PBF_TO_LOCAL = True        # sequential reads from local disk are faster/safer than FUSE for a ~2 GB file
NODE_CACHE = "flex_mem"         # libosmium node-location index; "sparse_file_array,/tmp/nodes.idx" trades RAM for disk

FQ = f"{CATALOG}.{SCHEMA}"
T = {n: f"{FQ}.{n}" for n in
     ("ref_road_category", "osm_road_way", "osm_admin_area", "dim_district", "dim_road", "fact_road_gmina")}
    
# snapshot label from the Geofabrik-style file name  poland-261101.osm.pbf -> 2026-11-01
_m = re.search(r"-(\d{2})(\d{2})(\d{2})\.osm\.pbf$", PBF_PATH)
OSM_SNAPSHOT = f"20{_m.group(1)}-{_m.group(2)}-{_m.group(3)}" if _m else os.path.basename(PBF_PATH)

print(f"target schema : {FQ}\nPBF           : {PBF_PATH}\nsnapshot      : {OSM_SNAPSHOT}\nre-extract    : {REEXTRACT}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1 · Constants (the "constant information")
# MAGIC
# MAGIC Line colours are the first slots of a colour-blind-checked categorical palette (blue / orange / aqua / violet); the road hierarchy is *also* encoded in line width, so colour is never the only cue.

# COMMAND ----------

# category, Polish name, English name, colour, line width (px), draw order (higher = on top)
ROAD_CATEGORIES = [
    ("A",  "Autostrada",        "Motorway",         "#2a78d6", 5.0, 4),
    ("S",  "Droga ekspresowa",  "Expressway",       "#eb6834", 4.0, 3),
    ("DK", "Droga krajowa",     "National road",    "#1baf7a", 3.0, 2),
    ("DW", "Droga wojewódzka",  "Voivodeship road", "#6250d6", 2.5, 1),
]
CATEGORY_NAME_PL = {c[0]: c[1] for c in ROAD_CATEGORIES}

# OSM highway=* values accepted as a road carriageway (links/ramps, construction, proposed are deliberately excluded)
ROAD_HIGHWAY_VALUES = {"motorway", "trunk", "primary", "secondary", "tertiary", "unclassified"}

ADMIN_LEVELS = {4: "voivodeship", 6: "powiat", 7: "gmina"}          # OSM admin_level -> our level name
TERC_LEN = {"voivodeship": 2, "powiat": 4, "gmina": 7}              # official TERYT code lengths
NAME_PREFIX = {"voivodeship": "wojewodztwo", "powiat": "powiat", "gmina": "gmina"}
LEVEL_ALIASES = {"wojewodztwo": "voivodeship", "woj": "voivodeship", "voivodeship": "voivodeship",
                 "powiat": "powiat", "gmina": "gmina"}
GMINA_TYPE = {"1": "urban", "2": "rural", "3": "urban-rural"}       # last digit of the 7-digit TERC
EXPECTED_COUNTS = {"voivodeship": 16, "powiat": 380, "gmina": 2477}  # 314 land + 66 city counties; used only for a sanity warning

VOIVODESHIP_CODES = {  # TERYT: sanity check for the OSM extraction
    "02": "dolnośląskie", "04": "kujawsko-pomorskie", "06": "lubelskie", "08": "lubuskie",
    "10": "łódzkie", "12": "małopolskie", "14": "mazowieckie", "16": "opolskie",
    "18": "podkarpackie", "20": "podlaskie", "22": "pomorskie", "24": "śląskie",
    "26": "świętokrzyskie", "28": "warmińsko-mazurskie", "30": "wielkopolskie", "32": "zachodniopomorskie"}

SIMPLIFY_ROAD_DEG = 0.0001      # ~8–11 m: invisible below zoom ~15, ~4–5x fewer vertices
SIMPLIFY_POLY_DEG = 0.0005      # ~40–55 m for district outlines

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2 · Helpers: ref parsing, name normalisation, geometry, Delta writer

# COMMAND ----------

_RE_REF = re.compile(r"^(?:(A|S)(\d{1,2})|(DK|DW)(\d{1,3}))$")
_RE_IGNORED_REF = re.compile(r"^(E\d{1,3}|\d{4,6}[A-Z])$")   # E-roads, powiat/gmina road numbers (e.g. 2101L)

In [0]:
def parse_ref_token(tok):
    """'A4'->'A4', 'S 5'->'S5', '94'->'DK94', '345'->'DW345', 'DK 12'->'DK12'; anything else -> None."""
    t = re.sub(r"[\s\-\.]", "", (tok or "").upper())
    m = _RE_REF.match(t)
    if m:
        cat, num = (m.group(1), m.group(2)) if m.group(1) else (m.group(3), m.group(4))
    elif t.isdigit() and len(t) <= 3:
        cat, num = ("DK" if len(t) <= 2 else "DW"), t
    else:
        return None
    return f"{cat}{int(num)}" if int(num) > 0 else None


def road_keys_from_ref(ref):
    """OSM ref may hold several values: 'A4;E40', '3; 24', 'S8;8'. Returns unique road keys in order."""
    out = []
    for tok in re.split(r"[;,]", ref or ""):
        k = parse_ref_token(tok)
        if k and k not in out:
            out.append(k)
    return out


def norm(s):
    """casefold + strip diacritics (incl. ł) + collapse punctuation: 'Województwo Dolnośląskie' -> 'wojewodztwo dolnoslaskie'."""
    s = (s or "").translate(str.maketrans("łŁ", "lL"))
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch)).casefold()
    return re.sub(r"[^0-9a-z]+", " ", s).strip()


def name_key(name, level):
    n = norm(name)
    p = NAME_PREFIX[level]
    return n[len(p) + 1:] if n.startswith(p + " ") else n


def terc_code(raw, level):
    """Normalise OSM teryt:terc to the canonical length for the level, or None."""
    d = re.sub(r"\D", "", raw or "")
    n = TERC_LEN[level]
    if len(d) == n:
        return d
    if len(d) == 7 and level != "gmina" and d[n:] == "0" * (7 - n):   # '0200000' style
        return d[:n]
    return None


def polygonal(g):
    """Return only the polygonal part of a (possibly repaired) geometry."""
    if g.geom_type in ("Polygon", "MultiPolygon"):
        return g
    parts = [p for p in shapely.get_parts(g) if p.geom_type in ("Polygon", "MultiPolygon")]
    return shapely.union_all(parts) if parts else g


def lines_only(arr):
    """From an array of intersection results keep LineString/MultiLineString parts; others (points, empty) -> None."""
    out = np.empty(len(arr), dtype=object)
    for i, (g, t) in enumerate(zip(arr, shapely.get_type_id(arr))):
        if t in (1, 5):
            out[i] = None if shapely.is_empty(g) else g
        elif t == 7:  # GeometryCollection: line + touching points
            parts = [p for p in shapely.get_parts(g) if p.geom_type in ("LineString", "MultiLineString")]
            out[i] = shapely.union_all(parts) if parts else None
        else:
            out[i] = None
    return out


_TR = {}


def _to_pl1992(geoms):

    if "t" not in _TR:
        _TR["t"] = pyproj.Transformer.from_crs(4326, 2180, always_xy=True)   # PL-1992: metric CRS for Poland

    def f(xy):
        x, y = _TR["t"].transform(xy[:, 0], xy[:, 1])
        return np.column_stack([x, y])
    return shapely.transform(geoms, f)


def length_m(geoms):
    return shapely.length(_to_pl1992(np.asarray(geoms, dtype=object)))


def area_km2(geoms):
    return shapely.area(_to_pl1992(np.asarray(geoms, dtype=object))) / 1e6


def write_delta(table, schema_ddl, rows, blob_col=None, max_bytes=24_000_000, max_rows=40_000):
    """Overwrite `table` with `rows` (list of tuples) in chunks small enough for Spark Connect's message limit."""
    state = {"first": True}

    def flush(buf):
        df = spark.createDataFrame(buf, schema=schema_ddl)
        w = df.write.mode("overwrite" if state["first"] else "append")
        if state["first"]:
            w = w.option("overwriteSchema", "true")
        w.saveAsTable(table)
        state["first"] = False

    buf, size = [], 0
    for r in rows:
        buf.append(r)
        size += (len(r[blob_col]) if blob_col is not None and r[blob_col] is not None else 0) + 200
        if size >= max_bytes or len(buf) >= max_rows:
            flush(buf)
            buf, size = [], 0
    if buf or state["first"]:
        flush(buf)
    print(f"  wrote {table}: {len(rows):,} rows")


def sql_rows(query):
    """THE data-access seam. In a Databricks App replace this one function with a databricks-sql-connector cursor."""
    return [r.asDict() for r in spark.sql(query).collect()]


def try_sql(stmt):
    try:
        spark.sql(stmt)
    except Exception as e:  # cosmetic / platform-specific statements must never break the pipeline
        print(f"  (skipped: {stmt[:60]}… -> {type(e).__name__})")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3 · Extract roads and administrative areas from the PBF (single `pyosmium` pass)
# MAGIC
# MAGIC `pyosmium` (C++ `libosmium`) is the only reader here that runs on serverless *and* assembles multipolygon/boundary relations (holes, enclaves, shared border ways).
# MAGIC The filters run in C++, so Python only sees the few hundred thousand objects we want:
# MAGIC * `with_areas(...)` filters apply in libosmium's first pass → only `boundary=administrative` relations with `admin_level` 4/6/7 are assembled.
# MAGIC * `KeyFilter("ref", "admin_level")` + `EntityFilter(WAY | AREA)` drop the ~300 M nodes and everything without a ref/level before it reaches Python.

# COMMAND ----------

def extract_pbf(pbf_path):
    

    local = None
    src = pbf_path
    if COPY_PBF_TO_LOCAL:
        local = tempfile.mkdtemp(prefix="osm_")
        src = os.path.join(local, "extract.osm.pbf")
        t0 = time.time()
        shutil.copyfile(pbf_path, src)
        print(f"copied PBF to local disk in {time.time() - t0:.0f}s ({os.path.getsize(src) / 1e6:,.0f} MB)")

    fp = (osmium.FileProcessor(src)
          .with_locations(NODE_CACHE)
          .with_areas(osmium.filter.TagFilter(("boundary", "administrative")),
                      osmium.filter.TagFilter(*[("admin_level", str(l)) for l in ADMIN_LEVELS]))
          .with_filter(osmium.filter.EntityFilter(osmium.osm.WAY | osmium.osm.AREA))
          .with_filter(osmium.filter.KeyFilter("ref", "admin_level")))
    wkb = osmium.geom.WKBFactory()

    ways, areas = [], []
    stats = Counter()
    unparsed = Counter()
    t0 = time.time()
    try:
        for o in fp:
            if o.is_way():
                hw = o.tags.get("highway")
                if hw not in ROAD_HIGHWAY_VALUES:
                    continue
                ref = o.tags.get("ref")
                keys = road_keys_from_ref(ref)
                if not keys:
                    if hw in ("motorway", "trunk", "primary", "secondary"):
                        for tok in re.split(r"[;,]", ref or ""):
                            tok = tok.strip()
                            if tok and not _RE_IGNORED_REF.match(re.sub(r"\s", "", tok.upper())):
                                unparsed[tok] += 1
                    continue
                try:
                    g = bytes.fromhex(wkb.create_linestring(o))
                except Exception:
                    stats["way_geometry_errors"] += 1
                    continue
                ways.append(dict(osm_way_id=o.id, highway=hw, ref_raw=ref, road_keys=keys,
                                 name=o.tags.get("name"), geom_wkb=g))
            elif o.is_area() and not o.from_way():
                try:
                    lvl = int(o.tags.get("admin_level", "0"))
                except ValueError:
                    continue
                if lvl not in ADMIN_LEVELS or o.tags.get("boundary") != "administrative":
                    continue
                try:
                    g = bytes.fromhex(wkb.create_multipolygon(o))
                except Exception:
                    stats["area_geometry_errors"] += 1
                    continue
                areas.append(dict(osm_rel_id=o.orig_id(), admin_level=lvl, name=o.tags.get("name"),
                                  teryt_terc=o.tags.get("teryt:terc"), geom_wkb=g))
    finally:
        if local:
            shutil.rmtree(local, ignore_errors=True)
    stats["seconds"] = round(time.time() - t0)
    print(f"extracted {len(ways):,} road ways and {len(areas):,} admin areas in {stats['seconds']}s; {dict(stats)}")
    return ways, areas, unparsed

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4 · Bronze, reference and silver builders

# COMMAND ----------

BRONZE_WAY_DDL = ("osm_way_id BIGINT, highway STRING, ref_raw STRING, road_keys ARRAY<STRING>, name STRING, "
                  "geom_wkb BINARY, osm_snapshot STRING")
BRONZE_AREA_DDL = ("osm_rel_id BIGINT, admin_level INT, name STRING, teryt_terc STRING, "
                   "geom_wkb BINARY, osm_snapshot STRING")
DIM_DISTRICT_DDL = ("level STRING, code STRING, name STRING, name_key STRING, label STRING, woj_code STRING, "
                    "pow_code STRING, parent_code STRING, gmina_type STRING, area_km2 DOUBLE, "
                    "center_lon DOUBLE, center_lat DOUBLE, min_lon DOUBLE, min_lat DOUBLE, max_lon DOUBLE, "
                    "max_lat DOUBLE, osm_rel_id BIGINT, geom_wkb BINARY, osm_snapshot STRING")
FACT_DDL = ("road_key STRING, category STRING, number INT, woj_code STRING, pow_code STRING, gmina_code STRING, "
            "length_m DOUBLE, min_lon DOUBLE, min_lat DOUBLE, max_lon DOUBLE, max_lat DOUBLE, "
            "geom_wkb BINARY, osm_snapshot STRING")
DIM_ROAD_DDL = ("road_key STRING, category STRING, number INT, label STRING, length_km DOUBLE, n_gminas INT, "
                "woj_codes ARRAY<STRING>, min_lon DOUBLE, min_lat DOUBLE, max_lon DOUBLE, max_lat DOUBLE, "
                "osm_snapshot STRING")


def write_reference():
    rows = [(c, pl, en, col, w, z) for c, pl, en, col, w, z in ROAD_CATEGORIES]
    write_delta(T["ref_road_category"],
                "category STRING, name_pl STRING, name_en STRING, color STRING, weight DOUBLE, z_order INT", rows)


def write_bronze(ways, areas):
    write_delta(T["osm_road_way"], BRONZE_WAY_DDL,
                [(w["osm_way_id"], w["highway"], w["ref_raw"], w["road_keys"], w["name"], w["geom_wkb"], OSM_SNAPSHOT)
                 for w in ways], blob_col=5)
    write_delta(T["osm_admin_area"], BRONZE_AREA_DDL,
                [(a["osm_rel_id"], a["admin_level"], a["name"], a["teryt_terc"], a["geom_wkb"], OSM_SNAPSHOT)
                 for a in areas], blob_col=4)


def load_bronze():
    ways = [dict(r, geom_wkb=bytes(r["geom_wkb"])) for r in sql_rows(f"SELECT * FROM {T['osm_road_way']}")]
    areas = [dict(r, geom_wkb=bytes(r["geom_wkb"])) for r in sql_rows(f"SELECT * FROM {T['osm_admin_area']}")]
    print(f"loaded bronze: {len(ways):,} road ways, {len(areas):,} admin areas")
    return ways, areas


def build_dim_district(areas):
    """-> (rows for dim_district, [(gmina_code, full-res polygon)] for the intersection step)"""
    best, skipped = {}, Counter()
    for a in areas:
        level = ADMIN_LEVELS[a["admin_level"]]
        code = terc_code(a["teryt_terc"], level)
        if code is None:
            skipped[level] += 1
            continue
        g = shapely.from_wkb(a["geom_wkb"])
        if not shapely.is_valid(g):
            g = polygonal(shapely.make_valid(g))
        ar = float(shapely.area(g))
        if (level, code) not in best or ar > best[(level, code)]["_area"]:      # duplicates: keep the biggest
            best[(level, code)] = dict(a, level=level, code=code, geom=g, _area=ar)
    if skipped:
        warnings.warn(f"admin areas without a usable teryt:terc were skipped: {dict(skipped)}")

    items = sorted(best.values(), key=lambda d: (d["level"] != "voivodeship", d["level"] != "powiat", d["code"]))
    simp = np.array([shapely.simplify(d["geom"], SIMPLIFY_POLY_DEG) for d in items], dtype=object)
    km2 = area_km2(simp)
    bounds = shapely.bounds(simp)
    centers = shapely.get_coordinates(shapely.point_on_surface(simp))
    present = {(d["level"], d["code"]) for d in items}

    rows, gminas = [], []
    for i, d in enumerate(items):
        lvl, code = d["level"], d["code"]
        parent = None if lvl == "voivodeship" else code[:2] if lvl == "powiat" else code[:4]
        if parent and (("voivodeship" if lvl == "powiat" else "powiat"), parent) not in present:
            warnings.warn(f"{lvl} {code} ({d['name']}) has no parent {parent} in the data")
        label = (d["name"] or code)[:1].upper() + (d["name"] or code)[1:]
        rows.append((lvl, code, d["name"], name_key(d["name"], lvl), label, code[:2],
                     code[:4] if lvl != "voivodeship" else None, parent,
                     GMINA_TYPE.get(code[6]) if lvl == "gmina" else None, float(km2[i]),
                     float(centers[i][0]), float(centers[i][1]), *[float(v) for v in bounds[i]],
                     int(d["osm_rel_id"]), shapely.to_wkb(simp[i]), OSM_SNAPSHOT))
        if lvl == "gmina":
            gminas.append((code, d["geom"]))

    cnt = Counter(r[0] for r in rows)
    for lvl, exp in EXPECTED_COUNTS.items():
        if cnt[lvl] != exp:
            warnings.warn(f"{lvl}: found {cnt[lvl]}, expected ~{exp} (OSM data changes slowly; check unless you know why)")
    missing_woj = sorted(set(VOIVODESHIP_CODES) - {r[1] for r in rows if r[0] == "voivodeship"})
    if missing_woj:
        warnings.warn(f"voivodeships missing from the extract: {missing_woj}")
    print(f"dim_district: {dict(cnt)}")
    return rows, gminas

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5 · Gold: clip every road to every gmina it touches
# MAGIC
# MAGIC Vectorised with shapely 2: an `STRtree` over the road ways, queried with the (prepared) gmina polygons.
# MAGIC Ways completely inside one gmina – the vast majority – skip the expensive `intersection`; only the few thousand boundary-crossing ways are cut.
# MAGIC Pieces are then merged per `(road_key, gmina)`, measured in EPSG:2180 (metres) *before* simplification, and stored simplified.

# COMMAND ----------

def build_fact_and_dim_road(ways, gminas):
    t0 = time.time()
    w_geoms = np.array([shapely.from_wkb(w["geom_wkb"]) for w in ways], dtype=object)
    g_codes = [c for c, _ in gminas]
    g_geoms = np.array([g for _, g in gminas], dtype=object)
    bad = ~shapely.is_valid(g_geoms)
    if bad.any():
        g_geoms[bad] = [polygonal(shapely.make_valid(g)) for g in g_geoms[bad]]
    shapely.prepare(g_geoms)                                    # prepared polygons: fast predicates

    tree = shapely.STRtree(w_geoms)
    gi, wi = tree.query(g_geoms, predicate="intersects")        # (gmina idx, way idx) candidate pairs
    inside = shapely.contains(g_geoms[gi], w_geoms[wi])
    clipped = np.empty(len(gi), dtype=object)
    clipped[inside] = w_geoms[wi[inside]]
    cut = ~inside
    if cut.any():
        clipped[cut] = lines_only(shapely.intersection(w_geoms[wi[cut]], g_geoms[gi[cut]]))
    print(f"pairs: {len(gi):,} ({int(inside.sum()):,} fully inside, {int(cut.sum()):,} cut)  [{time.time() - t0:.0f}s]")

    unmatched = len(ways) - len(np.unique(wi))
    groups = defaultdict(list)
    for g_idx, w_idx, geom in zip(gi, wi, clipped):
        if geom is None:
            continue
        for k in ways[w_idx]["road_keys"]:
            groups[(k, int(g_idx))].append(geom)

    keys = sorted(groups)
    merged = np.empty(len(keys), dtype=object)
    for i, key in enumerate(keys):
        merged[i] = shapely.line_merge(shapely.union_all(groups[key]))
    lens = length_m(merged)                                     # metres, from full-resolution geometry
    simp = shapely.simplify(merged, SIMPLIFY_ROAD_DEG)
    bnds = shapely.bounds(simp)
    wkbs = shapely.to_wkb(simp)

    fact = []
    for i, (rk, g_idx) in enumerate(keys):
        code = g_codes[g_idx]
        m = re.match(r"([A-Z]+)(\d+)$", rk)
        fact.append((rk, m.group(1), int(m.group(2)), code[:2], code[:4], code, float(lens[i]),
                     *[float(v) for v in bnds[i]], wkbs[i], OSM_SNAPSHOT))

    # dim_road: aggregate the fact rows
    agg = {}
    for r in fact:
        a = agg.setdefault(r[0], dict(cat=r[1], num=r[2], len=0.0, gm=0, woj=set(), b=[180, 90, -180, -90]))
        a["len"] += r[6]; a["gm"] += 1; a["woj"].add(r[3])
        a["b"] = [min(a["b"][0], r[7]), min(a["b"][1], r[8]), max(a["b"][2], r[9]), max(a["b"][3], r[10])]
    dim_road = [(k, a["cat"], a["num"], f"{CATEGORY_NAME_PL[a['cat']]} {k}", a["len"] / 1000.0, a["gm"],
                 sorted(a["woj"]), *a["b"], OSM_SNAPSHOT) for k, a in sorted(agg.items(), key=lambda kv: (kv[1]["cat"], kv[1]["num"]))]
    stats = dict(ways=len(ways), ways_outside_all_gminas=unmatched, fact_rows=len(fact), roads=len(dim_road),
                 seconds=round(time.time() - t0))
    print("gold:", stats)
    return fact, dim_road, stats

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6 · Run the pipeline

# COMMAND ----------

try_sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}")
write_reference()

unparsed_refs = Counter()
if REEXTRACT or not spark.catalog.tableExists(T["osm_road_way"]):
    ways, areas, unparsed_refs = extract_pbf(PBF_PATH)
    write_bronze(ways, areas)
else:
    ways, areas = load_bronze()

dim_rows, gminas = build_dim_district(areas)
write_delta(T["dim_district"], DIM_DISTRICT_DDL, dim_rows, blob_col=17)

fact_rows, dim_road_rows, gold_stats = build_fact_and_dim_road(ways, gminas)
write_delta(T["fact_road_gmina"], FACT_DDL, fact_rows, blob_col=11)
write_delta(T["dim_road"], DIM_ROAD_DDL, dim_road_rows)

# physical layout / governance (cosmetic, safe to fail)
try_sql(f"ALTER TABLE {T['fact_road_gmina']} CLUSTER BY (road_key, woj_code)")
try_sql(f"OPTIMIZE {T['fact_road_gmina']}")
for t, c in {"ref_road_category": "Render style per road category (A/S/DK/DW)",
             "osm_road_way": "Bronze: OSM ways with A/S/DK/DW ref, full-resolution WKB (EPSG:4326)",
             "osm_admin_area": "Bronze: OSM boundary polygons admin_level 4/6/7, full-resolution WKB (EPSG:4326)",
             "dim_district": "Voivodeships, powiats, gminas keyed by TERYT code; simplified outline WKB (EPSG:4326)",
             "dim_road": "One row per road key (A4, S5, DK94, DW345)",
             "fact_road_gmina": "Each road clipped to each gmina it passes through; simplified WKB (EPSG:4326). "
                                "Filter by road_key + woj_code/pow_code/gmina_code to render a road within a district."}.items():
    try_sql(f"COMMENT ON TABLE {T[t]} IS '{c} | source: OpenStreetMap contributors (ODbL), snapshot {OSM_SNAPSHOT}'")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7 · Sanity checks
# MAGIC Things worth a look after the first run on the real file: counts per level, roads per category, and the refs that *looked* like roads but did not parse (to spot tagging variants).

# COMMAND ----------

def _show(rows, title):
    print(f"\n== {title}")
    for r in rows:
        print("  ", r)

_show(sql_rows(f"SELECT level, count(*) AS n FROM {T['dim_district']} GROUP BY level ORDER BY level"), "districts per level")
_show(sql_rows(f"SELECT category, count(*) AS roads, round(sum(length_km)) AS km FROM {T['dim_road']} GROUP BY category ORDER BY category"),
      "roads per category (km counts each carriageway separately)")
_show(sql_rows(f"SELECT woj_code, round(sum(length_m)/1000, 1) AS km FROM {T['fact_road_gmina']} WHERE road_key = 'A4' GROUP BY woj_code ORDER BY woj_code"),
      "A4 length per voivodeship code")
print("\n== ways that intersect no gmina (outside Poland / border stubs):", gold_stats["ways_outside_all_gminas"])
if unparsed_refs:
    _show(unparsed_refs.most_common(15), "most common NOT-parsed refs on motorway/trunk/primary/secondary (extend parse_ref_token if you see real roads here)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8 · Query layer + `render_roads_map()`
# MAGIC
# MAGIC Request format (exactly your example): list of `{road_key: [districts]}`. A district is `"Województwo Dolnośląskie"`, `"Powiat Wrocław"`, `"Gmina Oława"` (case- and diacritics-insensitive) **or a TERYT code** (`"02"`, `"0264"`, `"0264011"`).
# MAGIC Several districts for one road are a **union** ("A4 inside Dolnośląskie *or* Opolskie"); an empty list / `None` means the whole road.
# MAGIC Ambiguous names (Poland has e.g. both an urban and a rural *gmina Oława*) raise an error that lists the TERYT codes to choose from.
# MAGIC
# MAGIC `get_road_geojson()` has **no UI dependency** – it is the function to lift into the Databricks App; `render_roads_map()` is a thin folium wrapper on top.

# COMMAND ----------

class RoadMapError(ValueError):
    pass


_CACHE = {}


def load_catalog(refresh=False):
    """Small lookup tables (≈3k districts, ≈1k roads, 4 categories). Cache in the app process; reload with refresh=True."""
    if refresh or "districts" not in _CACHE:
        _CACHE["districts"] = sql_rows(
            f"SELECT level, code, name, name_key, label, woj_code, pow_code, parent_code, gmina_type FROM {T['dim_district']}")
        _CACHE["roads"] = {r["road_key"]: r for r in sql_rows(
            f"SELECT road_key, category, number, label, length_km FROM {T['dim_road']}")}
        _CACHE["styles"] = {r["category"]: r for r in sql_rows(
            f"SELECT category, name_pl, name_en, color, weight, z_order FROM {T['ref_road_category']}")}
    return _CACHE


def normalize_road_key(key):
    t = re.sub(r"[\s\-_\.]", "", str(key).upper())
    m = re.fullmatch(r"(A|S|DK|DW)(\d{1,3})", t)
    if not m:
        raise RoadMapError(f"'{key}' is not a road key. Use category + number, e.g. 'A4', 'S5', 'DK94', 'DW345'.")
    return f"{m.group(1)}{int(m.group(2))}"


def resolve_district(spec, cat):
    """'Województwo Dolnośląskie' | 'Powiat Wrocław' | 'Gmina Oława' | '02' | '0264' | '0264011' -> dim_district row."""
    s = str(spec).strip()
    pool = cat["districts"]
    if re.fullmatch(r"\d{2}|\d{4}|\d{7}", s):
        hit = [d for d in pool if d["code"] == s]
        if not hit:
            raise RoadMapError(f"No district with TERYT code {s}.")
        return hit[0]
    toks = norm(s).split(" ")
    level = LEVEL_ALIASES.get(toks[0]) if toks and toks[0] in LEVEL_ALIASES else None
    key = " ".join(toks[1:]) if level else " ".join(toks)
    cands = [d for d in pool if d["name_key"] == key and (level is None or d["level"] == level)]
    if len(cands) == 1:
        return cands[0]
    if not cands:
        near = difflib.get_close_matches(key, [d["name_key"] for d in pool if level is None or d["level"] == level], n=5, cutoff=0.7)
        raise RoadMapError(f"District '{spec}' not found." + (f" Did you mean: {sorted(set(near))}?" if near else ""))
    names = {d["code"]: d for d in pool}
    opts = "; ".join(f"{d['code']} = {d['level']} '{d['name']}'"
                     + (f" in {names[d['parent_code']]['name']}" if d["parent_code"] in names else "") for d in cands[:8])
    raise RoadMapError(f"'{spec}' is ambiguous ({len(cands)} matches). Pass the TERYT code instead: {opts}")


def parse_request(road_districts, cat):
    items = road_districts.items() if isinstance(road_districts, dict) else [kv for d in road_districts for kv in d.items()]
    reqs = []
    for key, dists in items:
        rk = normalize_road_key(key)
        if rk not in cat["roads"]:
            near = difflib.get_close_matches(rk, list(cat["roads"]), n=5, cutoff=0.6)
            raise RoadMapError(f"Road {rk} not found in this OSM snapshot." + (f" Similar roads: {near}" if near else ""))
        if dists is None or isinstance(dists, str):
            dists = [] if dists is None else [dists]
        reqs.append(dict(road_key=rk, districts=[resolve_district(d, cat) for d in dists]))
    return reqs


def _segments_sql(rq):
    conds = []
    for col, level in (("woj_code", "voivodeship"), ("pow_code", "powiat"), ("gmina_code", "gmina")):
        codes = sorted({d["code"] for d in rq["districts"] if d["level"] == level})
        if codes:
            assert all(c.isdigit() for c in codes)               # codes come from dim_district, never from raw user text
            conds.append(f"{col} IN ({', '.join(repr(c) for c in codes)})")
    where = f"road_key = '{rq['road_key']}'" + (f" AND ({' OR '.join(conds)})" if conds else "")
    return f"SELECT length_m, geom_wkb FROM {T['fact_road_gmina']} WHERE {where}"


def _geojson(g, nd=5):
    g = shapely.transform(g, lambda xy: np.round(xy, nd))
    return json.loads(shapely.to_geojson(g))


def get_road_geojson(road_districts, with_districts=True, refresh_catalog=False):
    """-> dict(roads=FeatureCollection, districts=FeatureCollection, bbox=[minx,miny,maxx,maxy], missing=[...]). No UI dependency."""
    cat = load_catalog(refresh_catalog)
    reqs = parse_request(road_districts, cat)
    feats, missing, boxes = [], [], []
    for rq in reqs:
        rows = sql_rows(_segments_sql(rq))
        names = ", ".join(d["label"] for d in rq["districts"]) or "whole road"
        if not rows:
            missing.append(f"{rq['road_key']} in {names}: the road does not pass through these districts")
            continue
        merged = shapely.line_merge(shapely.union_all([shapely.from_wkb(bytes(r["geom_wkb"])) for r in rows]))
        boxes.append(merged.bounds)
        c = cat["roads"][rq["road_key"]]["category"]
        st = cat["styles"][c]
        feats.append({"type": "Feature", "geometry": _geojson(merged),
                      "properties": dict(road_key=rq["road_key"], category=c, category_name=st["name_pl"], districts=names,
                                         length_km=round(sum(r["length_m"] for r in rows) / 1000.0, 1),
                                         color=st["color"], weight=st["weight"], z_order=st["z_order"])})
    feats.sort(key=lambda f: f["properties"]["z_order"])         # big roads drawn last (on top)

    dfeats = []
    if with_districts:
        codes = sorted({d["code"] for rq in reqs for d in rq["districts"]})
        if codes:
            for r in sql_rows(f"SELECT code, label, level, geom_wkb FROM {T['dim_district']} WHERE code IN ({', '.join(repr(c) for c in codes)})"):
                g = shapely.from_wkb(bytes(r["geom_wkb"]))
                boxes.append(g.bounds)
                dfeats.append({"type": "Feature", "geometry": _geojson(g),
                               "properties": dict(code=r["code"], label=r["label"], level=r["level"])})
    for m in missing:
        warnings.warn(m)
    if not feats:
        raise RoadMapError("Nothing to draw: " + "; ".join(missing))
    bbox = [min(b[0] for b in boxes), min(b[1] for b in boxes), max(b[2] for b in boxes), max(b[3] for b in boxes)]
    return dict(roads={"type": "FeatureCollection", "features": feats},
                districts={"type": "FeatureCollection", "features": dfeats}, bbox=bbox, missing=missing)


def render_roads_map(road_districts, show_districts=True, tiles="CartoDB positron"):
    """Interactive folium map. In a notebook:  displayHTML(render_roads_map([...]).get_root().render())"""
    
    data = get_road_geojson(road_districts, with_districts=show_districts)
    x0, y0, x1, y1 = data["bbox"]
    m = folium.Map(location=[(y0 + y1) / 2, (x0 + x1) / 2], tiles=tiles, control_scale=True, zoom_start=7)
    if data["districts"]["features"]:
        folium.GeoJson(data["districts"], name="Districts",
                       style_function=lambda f: {"color": "#52514e", "weight": 1.2, "dashArray": "4 4", "fillOpacity": 0.04},
                       tooltip=folium.GeoJsonTooltip(fields=["label"], aliases=[""])).add_to(m)
    folium.GeoJson(data["roads"], name="Roads",
                   style_function=lambda f: {"color": f["properties"]["color"], "weight": f["properties"]["weight"], "opacity": 0.95},
                   highlight_function=lambda f: {"weight": f["properties"]["weight"] + 2},
                   tooltip=folium.GeoJsonTooltip(fields=["road_key", "category_name", "districts", "length_km"],
                                                 aliases=["Road", "Type", "Within", "Length [km]"])).add_to(m)
    cats = sorted({f["properties"]["category"] for f in data["roads"]["features"]},
                  key=lambda c: -load_catalog()["styles"][c]["z_order"])
    legend = "".join(f'<div><span style="display:inline-block;width:22px;height:{load_catalog()["styles"][c]["weight"]}px;'
                     f'background:{load_catalog()["styles"][c]["color"]};vertical-align:middle;margin-right:6px"></span>'
                     f'{c} – {load_catalog()["styles"][c]["name_pl"]}</div>' for c in cats)
    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:rgba(255,255,255,.92);padding:8px 10px;'
        'font:12px/1.5 sans-serif;color:#0b0b0b;border-radius:6px;box-shadow:0 1px 4px rgba(0,0,0,.3)">' + legend + '</div>'))
    m.fit_bounds([[y0, x0], [y1, x1]])
    return m

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9 · Try it

# COMMAND ----------

road_districts = [
    {"A4": ["Województwo Dolnośląskie", "Województwo Opolskie"]},
    {"DK94": ["Województwo Dolnośląskie", "Powiat Wrocław"]},
]

m = render_roads_map(road_districts)
try:
    displayHTML(m.get_root().render())
except NameError:                       # outside Databricks
    m.save(os.environ.get("MAP_HTML", "roads_map.html"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10 · Taking this into a Databricks App
# MAGIC
# MAGIC * Copy section 8 (`load_catalog` … `get_road_geojson`, and the constants/`norm` helpers they use) into the app. **Replace only `sql_rows()`** with a `databricks-sql-connector` call (`cursor.execute(query)` → list of dicts, using `Config()` credentials as in the Databricks Apps Streamlit tutorial). Nothing else in the query layer is Spark-specific.
# MAGIC * App service principal needs `SELECT` on the five tables and `CAN USE` on one SQL warehouse. App requirements: `shapely`, `numpy`, `databricks-sql-connector`, `databricks-sdk` (+ `streamlit-folium` / `folium`, or hand the GeoJSON from `get_road_geojson()` to Leaflet/pydeck in the browser).
# MAGIC * Load `load_catalog()` once per process (`st.cache_resource`). `fact_road_gmina` is small enough to cache entirely in the app if you prefer zero warehouse round-trips per click.
# MAGIC * Want SQL-side spatial work instead? Everything is WKB: `ST_GeomFromWKB(geom_wkb, 4326)` gives Databricks `GEOMETRY` and `ST_AsGeoJSON(...)` lets the warehouse return GeoJSON directly.
# MAGIC * Re-run this notebook (with the new `pbf_path`) whenever you load a fresh extract; `osm_snapshot` is stored on every row.